# Precon card retention

Reads the already-populated `precon_card_retention` table (`edhcut.ingest.precon_retention`) — no network calls, no re-derivation. For each `(precon, commander)` group, shows a bar chart of each precon card's *weighted cut rate*: `weighted_cut / (weighted_cut + weighted_kept)`.

Unlike a plain "was it cut" count, every deck matching that precon's commander(s) contributes here — regardless of how much it diverged from the precon — weighted by `edhcut.ingest.precon_retention.cut_confidence(diff)`. A deck that barely touched the precon gives strong evidence about the few cards it *did* cut and weak evidence about the untouched rest (inertia, not a conscious keep); a heavily rebuilt deck is the mirror image — weak evidence per individual cut (everything was in flux anyway) but strong evidence for whatever specific cards survived a near-total rebuild. No hard similarity cutoff, so even commanders with very few near-exact-copy decks (or none at all) still get a real signal.

**`weighted_cut + weighted_kept` is the group's effective sample size** (every deck contributes exactly 1.0 total weight, split between the two) — shown in the chart title and hover, so a rate built on a thin precon-commander match doesn't read as more confident than it is.

In [ ]:
import pandas as pd
import plotly.express as px

from edhcut.config import CONFIG
from edhcut.db import connect

pd.set_option("display.max_colwidth", 60)

# Keep the context-manager object itself alive (not just the yielded `conn`) — otherwise the
# generator-based context manager can get garbage-collected mid-notebook, which runs its own
# `finally: conn.close()` and closes the connection out from under later cells.
_db_ctx = connect(CONFIG.paths.db_path)
conn = _db_ctx.__enter__()

In [ ]:
def commander_label(conn, commander_key: str) -> str:
    """commander_key is a bare oracle_id for a single commander, \"id1+id2\" for a partner
    pair (same convention as edhrec_card_stats.commander_key) — resolve each half to a name."""
    names = []
    for oracle_id in commander_key.split("+"):
        row = conn.execute("SELECT name FROM cards WHERE oracle_id = ?", (oracle_id,)).fetchone()
        names.append(row[0] if row else oracle_id)
    return " + ".join(names)


df = pd.read_sql_query(
    """
    SELECT r.precon_id, r.commander_key, r.oracle_id, c.name AS card_name,
           r.similar_deck_count, r.kept_count, r.weighted_cut, r.weighted_kept,
           p.deck_name AS precon_name
    FROM precon_card_retention r
    JOIN cards c ON c.oracle_id = r.oracle_id
    JOIN precons p ON p.precon_id = r.precon_id
    """,
    conn,
)
df["weighted_total"] = df["weighted_cut"] + df["weighted_kept"]
df["weighted_cut_rate"] = df["weighted_cut"] / df["weighted_total"]
df["commander_name"] = df["commander_key"].map(lambda k: commander_label(conn, k))

print(f"{len(df)} rows across {df.groupby(['precon_id', 'commander_key']).ngroups} (precon, commander) group(s).")
df.sort_values("weighted_cut_rate", ascending=False).head(10)

155 rows across 2 (precon, commander) group(s).


,precon_id,commander_key,oracle_id,card_name,similar_deck_count,kept_count,weighted_cut,weighted_kept,precon_name,weighted_total,weighted_cut_rate,commander_name
6,22357696,68418069-f615-40ef-ae0d-764192acae00,bc59977f-b3a7-4bc1-9f20-0a1e36d418ae,Spreading Insurrection,0,0,9.777109,0.000000,Goblin Storm Secret Lair Commander,9.777109,1.000000,"Krenko, Mob Boss"
18,22357696,68418069-f615-40ef-ae0d-764192acae00,e05abb6b-e9f4-4d9e-ad1e-7805ebc09914,Fists of Flame,0,0,9.777109,0.000000,Goblin Storm Secret Lair Commander,9.777109,1.000000,"Krenko, Mob Boss"
17,22357696,68418069-f615-40ef-ae0d-764192acae00,61978202-538b-4f92-acfc-e95f4a6ce466,Wild Ride,0,0,9.777109,0.000000,Goblin Storm Secret Lair Commander,9.777109,1.000000,"Krenko, Mob Boss"
23,22357696,68418069-f615-40ef-ae0d-764192acae00,a95a0b13-d50f-41cf-8668-de60d445e7b0,Ancestors' Aid,0,0,9.777109,0.000000,Goblin Storm Secret Lair Commander,9.777109,1.000000,"Krenko, Mob Boss"
77,22357696,68418069-f615-40ef-ae0d-764192acae00,37a18736-5fe2-4897-809b-013497bdd890,Past in Flames,0,0,9.777109,0.000000,Goblin Storm Secret Lair Commander,9.777109,1.000000,"Krenko, Mob Boss"
65,22357696,68418069-f615-40ef-ae0d-764192acae00,f17d0fb8-c157-43b8-be26-f5ba4c6aed14,Haze of Rage,0,0,9.777109,0.000000,Goblin Storm Secret Lair Commander,9.777109,1.000000,"Krenko, Mob Boss"
54,22357696,68418069-f615-40ef-ae0d-764192acae00,4c340b82-22e1-445a-b236-82471b442031,Frontline Heroism,0,0,9.777109,0.000000,Goblin Storm Secret Lair Commander,9.777109,1.000000,"Krenko, Mob Boss"
134,2209041,726cd041-5d0b-436c-bced-9335f56c0b0d,874e0b54-bc24-4887-abe1-1ecfd3a3abae,Trostani's Summoner,5,1,16.222253,0.141851,Coven Counters - Midnight Hunt Commander,16.364104,0.991332,"Kyler, Sigardian Emissary"
109,2209041,726cd041-5d0b-436c-bced-9335f56c0b0d,d063f1d4-eb6b-40fa-97b7-f15ea53397fb,Curse of Clinging Webs,5,1,15.933202,0.852801,Coven Counters - Midnight Hunt Commander,16.786003,0.949196,"Kyler, Sigardian Emissary"
106,2209041,726cd041-5d0b-436c-bced-9335f56c0b0d,ef08d371-9d6c-4a2d-a444-d475fbf1b633,Bestial Menace,5,1,16.024437,0.944035,Coven Counters - Midnight Hunt Commander,16.968472,0.944365,"Kyler, Sigardian Emissary"


## Weighted cut rate per precon commander

In [ ]:
if df.empty:
    print("No precon_card_retention rows yet — run `python -m edhcut.ingest.precon_retention` first.")

for (precon_id, commander_key), group in df.groupby(["precon_id", "commander_key"]):
    commander_name = group["commander_name"].iloc[0]
    precon_name = group["precon_name"].iloc[0]
    effective_n = group["weighted_total"].iloc[0]
    sorted_group = group.sort_values("weighted_cut_rate", ascending=False)

    fig = px.bar(
        sorted_group,
        x="card_name",
        y="weighted_cut_rate",
        hover_data=["weighted_cut", "weighted_kept", "kept_count", "similar_deck_count"],
        title=f"{commander_name} — weighted cut rate vs '{precon_name}' \n(effective n\u2248{effective_n:.0f} decks)",
        labels={"weighted_cut_rate": "weighted cut rate", "card_name": ""},
    )
    fig.update_layout(yaxis_tickformat=".0%", xaxis_tickangle=-65, height=500)
    fig.show()